### Preprocessing

Load Dataset - Enron

In [0]:
dbutils.fs.ls("/FileStore/tables/")

[FileInfo(path='dbfs:/FileStore/tables/BDTT_Assignment_1_Enron/', name='BDTT_Assignment_1_Enron/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/BDTT_Assignment_1_Enron.zip', name='BDTT_Assignment_1_Enron.zip', size=375294957, modificationTime=1739229487000),
 FileInfo(path='dbfs:/FileStore/tables/flood.csv', name='flood.csv', size=128984, modificationTime=1739637363000),
 FileInfo(path='dbfs:/FileStore/tables/iotstream/', name='iotstream/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/iotstream.zip', name='iotstream.zip', size=43891, modificationTime=1740066152000),
 FileInfo(path='dbfs:/FileStore/tables/test-1.json', name='test-1.json', size=17958, modificationTime=1739113851000),
 FileInfo(path='dbfs:/FileStore/tables/test.json', name='test.json', size=17958, modificationTime=1739113480000),
 FileInfo(path='dbfs:/FileStore/tables/webpage/', name='webpage/', size=0, modificationTime=0),
 FileInfo(path='dbfs:/FileStore/tables/webpage.zip', 

Copy zip folder to DBFS

In [0]:
dbutils.fs.cp("/FileStore/tables/BDTT_Assignment_1_Enron.zip", "file:/tmp/")

True

In [0]:
%sh
ls /tmp/

BDTT_Assignment_1_Enron.zip
Rserv
Rtmpnk8EnA
chauffeur-daemon-params
chauffeur-daemon.pid
chauffeur-env.sh
custom-spark.conf
driver-daemon-params
driver-daemon.pid
driver-env.sh
hsperfdata_root
systemd-private-e64eb3766ff34ba8a69eada4abc28173-systemd-logind.service-oJGCNF
systemd-private-e64eb3766ff34ba8a69eada4abc28173-systemd-resolved.service-UM65ub
tmp.mVpdbWpDzY


Unzip dataset

In [0]:
%sh
unzip -d /tmp/ /tmp/BDTT_Assignment_1_Enron.zip

Archive:  /tmp/BDTT_Assignment_1_Enron.zip
  inflating: /tmp/emails.csv         


In [0]:
%sh
ls /tmp/emails.csv

/tmp/emails.csv


Create folder in DBFS and move the files from Local node to DBFS

In [0]:
dbutils.fs.mkdirs("FileStore/tables/BDTT_Assignment_1_Enron")

True

In [0]:
dbutils.fs.mv("file:/tmp/emails.csv", "/FileStore/tables/BDTT_Assignment_1_Enron", True)

True

In [0]:
dbutils.fs.ls("FileStore/tables/BDTT_Assignment_1_Enron/")

[FileInfo(path='dbfs:/FileStore/tables/BDTT_Assignment_1_Enron/emails.csv', name='emails.csv', size=1426122219, modificationTime=1740819784000)]

Display Dataset Head - 600 bytes

In [0]:
dbutils.fs.head("FileStore/tables/BDTT_Assignment_1_Enron/emails.csv", 600)

[Truncated to first 600 bytes]


'"file","message"\n"allen-p/_sent_mail/1.","Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Tim Belden <Tim Belden/Enron@EnronXGate>\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pallen (Non-Privileged).pst\n\nHere is our forecast\n\n "\n"allen-p/_sent_mail/10.","Message-ID: <15464986.1075855378456.Ja'

In [0]:
for line in dbutils.fs.head('FileStore/tables/BDTT_Assignment_1_Enron/emails.csv', 600).splitlines():
    print(line)

[Truncated to first 600 bytes]
"file","message"
"allen-p/_sent_mail/1.","Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 "
"allen-p/_sent_mail/10.","Message-ID: <15464986.1075855378456.Ja


Reading Dataset as per Assessment Brief

In [0]:
df = spark.read.csv(
"/FileStore/tables/BDTT_Assignment_1_Enron/emails.csv",
header=True, # Use the first row as the header
inferSchema=True, # Infer data types
quote='"', # Define the quote character
escape='"', # Escape quotes inside quoted fields
multiLine=True # Enable multiline support
)

In [0]:
 df.head()

Row(file='allen-p/_sent_mail/1.', message="Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Tim Belden <Tim Belden/Enron@EnronXGate>\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pallen (Non-Privileged).pst\n\nHere is our forecast\n\n ")

There are Two Columns File and Message

In [0]:
df.columns

['file', 'message']

In [0]:
rdd = df.select("message").rdd.map(lambda row: row["message"] if row["message"] else "")


In [0]:
rdd.take(2)


["Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Tim Belden <Tim Belden/Enron@EnronXGate>\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pallen (Non-Privileged).pst\n\nHere is our forecast\n\n ",
 "Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>\nX-cc: \nX-bcc: \nX-Folder: \\Phillip_Allen_Jan2002_1\\Allen, Phillip K.\\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pa

### **Email Metadata Extraction Using Regex in PySpark**  

To analyze email data efficiently, a **regular expression (regex)** approach was implemented to extract key metadata fields such as `From`, `To`, `Subject`, and `Date` from raw email headers. Regex enables structured extraction by identifying patterns in text data, making it suitable for processing unstructured email content.  

#### **Methodology:**  
1. **Dataset Analysis:** The dataset was examined to identify key email headers.  
2. **Regex Pattern Definition:** A dictionary of regex patterns was created to match each metadata field.  
3. **Data Processing with PySpark:**  
   - The dataset was processed using **RDD transformations**.  
   - The `map()` function applied regex to extract metadata from each email.  
   - Missing values were handled by assigning empty strings where no match was found.  

**Next Step?:** The parsed email metadata will be converted into a **PySpark DataFrame** for further analysis.  

In [0]:
import re
from pyspark.sql import functions as F

# Email header patterns
patterns = {
    "Message-ID": r"Message-ID: <([^>]+)>",
    "Date": r"Date: (.*?)(?:\n|$)",
    "From": r"From: (.*?)(?:\n|$)",
    "To": r"To: (.*?)(?:\n|$)",
    "Subject": r"Subject: (.*?)(?:\n|$)",
    "Mime-Version": r"Mime-Version: (.*?)(?:\n|$)",
    "Content-Type": r"Content-Type: (.*?)(?:\n|$)",
    "Content-Transfer-Encoding": r"Content-Transfer-Encoding: (.*?)(?:\n|$)",
    "X-From": r"X-From: (.*?)(?:\n|$)",
    "X-To": r"X-To: (.*?)(?:\n|$)",
    "X-cc": r"X-cc: (.*?)(?:\n|$)",
    "X-bcc": r"X-bcc: (.*?)(?:\n|$)",
    "X-Folder": r"X-Folder: (.*?)(?:\n|$)",
    "X-Origin": r"X-Origin: (.*?)(?:\n|$)",
    "X-FileName": r"X-FileName: (.*?)(?:\n|$)",
}

# Parsing each email
parsed_rdd = rdd.map(lambda email_data: {
    key: re.search(pattern, email_data).group(1).strip() if re.search(pattern, email_data) else ""
    for key, pattern in patterns.items()
})

# Example: Collecting results (you can adjust how you want to process it, such as converting to a DataFrame)
parsed_email_list = parsed_rdd.collect()


In [0]:
# Convert parsed_rdd to DataFrame
df = spark.createDataFrame(parsed_rdd)


### Exploratory Data Analysis

In [0]:
# 1. Check column names
print("Columns in the DataFrame:", df.columns)

# 2. Show total number of rows
total_entries = df.count()
print(f"Total entries in the DataFrame: {total_entries}")

# 3. Display the first 5 rows
df.show(5)

# 4. Check for null or missing values in each column
print("Null values in each column:")
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# 5. Summary statistics for numeric columns
print("Summary statistics for numeric columns:")
df.describe().show()

# 6. Check data types of each column
print("Data types of each column:", df.dtypes)

Columns in the DataFrame: ['Content-Transfer-Encoding', 'Content-Type', 'Date', 'From', 'Message-ID', 'Mime-Version', 'Subject', 'To', 'X-FileName', 'X-Folder', 'X-From', 'X-Origin', 'X-To', 'X-bcc', 'X-cc']
Total entries in the DataFrame: 517401
+-------------------------+--------------------+--------------------+--------------------+--------------------+------------+---------+--------------------+--------------------+--------------------+---------------+--------+--------------------+-----+----+
|Content-Transfer-Encoding|        Content-Type|                Date|                From|          Message-ID|Mime-Version|  Subject|                  To|          X-FileName|            X-Folder|         X-From|X-Origin|                X-To|X-bcc|X-cc|
+-------------------------+--------------------+--------------------+--------------------+--------------------+------------+---------+--------------------+--------------------+--------------------+---------------+--------+--------------------+

### Group A - Question 3:  What is the number of emails with a subject line (i.e., the 'Subject' field is not missing or empty)?

In [0]:
from pyspark.sql.functions import col

df.select(col("Message-ID"), col("Subject")).show(10, truncate=False)

+-------------------------------------------+-------------------------------------------------------+
|Message-ID                                 |Subject                                                |
+-------------------------------------------+-------------------------------------------------------+
|18782981.1075855378110.JavaMail.evans@thyme|                                                       |
|15464986.1075855378456.JavaMail.evans@thyme|Re:                                                    |
|24216240.1075855687451.JavaMail.evans@thyme|Re: test                                               |
|13505866.1075863688222.JavaMail.evans@thyme|                                                       |
|30922949.1075863688243.JavaMail.evans@thyme|Re: Hello                                              |
|30965995.1075863688265.JavaMail.evans@thyme|Re: Hello                                              |
|16254169.1075863688286.JavaMail.evans@thyme|                                     

As it can be seen through visualizing a sample of 20 emails subjects, there can be a few ways for a subject to empty:

- an empty subject, that has no charecter.
- a reply or forward of emails with no subjcet, this is were the subject line contains either "Re:" or "FW:", which means relpying or forwarding an email with no subject, for this reason we have counted these subjects as empty as well.

In [0]:
from pyspark.sql.functions import col, lower

# Filtering and counting rows
count = df.filter(
   ~ lower(col("Subject")).isin(["", "re:", "fw:", "fwd:"])
).count()

print(count)

484396


### Group B - Question 1: Who are the top 10 senders by number of emails sent (based on the ‘From’ field), along with the number of emails they each sent

In [0]:
from pyspark.sql.functions import col, count, desc

top_senders = (
    df.filter(col("From").isNotNull())  # Ensure 'From' field is not null
    .groupBy("From")                    # Group by sender
    .agg(count("*").alias("email_count")) # Count emails per sender
    .orderBy(desc("email_count"))        # Sort in descending order
    .limit(10)                           # Get top 10 senders
)
print("Top 10 senders by number of emails sent")
top_senders.show()

Top 10 senders by number of emails sent
+--------------------+-----------+
|                From|email_count|
+--------------------+-----------+
|  kay.mann@enron.com|      16735|
|vince.kaminski@en...|      14368|
|jeff.dasovich@enr...|      11411|
|pete.davis@enron.com|       9149|
|chris.germany@enr...|       8801|
|sara.shackleton@e...|       8777|
|enron.announcemen...|       8587|
|tana.jones@enron.com|       8490|
|steven.kean@enron...|       6759|
|kate.symes@enron.com|       5438|
+--------------------+-----------+



### Group C - Question 2: Who are the top ten senders (based on the 'From’ field) who received no emails themselves (based on the 'To’ field)

To identify top senders who did not receive any emails, two datasets were created: one for senders with email counts and another for recipients.  

1. **Sender Data**: Emails were grouped by sender to compute the email count.  
2. **Recipient Data**: The "To" column, containing lists of recipients, was exploded into distinct rows. A distinct filter was applied to retain only unique recipients.   



In [0]:
%python

#Count email sent per senders (Exclusing null 'From' values)
senders_df = df.filter(F.col('From').isNotNull()).groupBy('From') \
    .count().withColumnRenamed('count', 'sent_count')

#Extract unique recipients from 'To' (handling multiple recipients and nulls)
recipients_df = df.filter(F.col('To').isNotNull()) \
    .withColumn('recipient', F.explode(F.split(F.col('To'), ',\\s*'))) \
        .select(F.col('recipient').alias('email')) \
            .distinct()

print(senders_df.head(3))
print(recipients_df.head(3))

[Row(From='anchordesk_daily@anchordesk.zdlists.com', sent_count=20), Row(From='feedback@intcx.com', sent_count=607), Row(From='contact@dew-consulting.com', sent_count=4)]
[Row(email='john.lavorato@enron.com'), Row(email='leah.arsdall@enron.com'), Row(email='tim.belden@enron.com')]


3. **Analysis**: A left anti join was performed between senders and recipients to isolate senders who never received an email. The results were ordered by the highest sent count, displaying the top 10 senders with no received emails. 

In [0]:
#Senders who never recieved an email (Left Anti Join)
senders_not_recipients_df = senders_df.join(recipients_df, senders_df['From']\
    == recipients_df['email'], 'left_anti')

#Get the top 10 senders who never recieved email, sorted by sent count
top_10_senders = senders_not_recipients_df.orderBy(F.desc('sent_count')) \
    .limit(10)

#Display result
top_10_senders.show(truncate=False)


+------------------------------------------+----------+
|From                                      |sent_count|
+------------------------------------------+----------+
|no.address@enron.com                      |5112      |
|noreply@ccomad3.uu.commissioner.com       |877       |
|owner-nyiso_tech_exchange@lists.thebiz.net|712       |
|owner-eveningmba@haas.berkeley.edu        |508       |
|exchange.administrator@enron.com          |455       |
|wsmith@wordsmith.org                      |454       |
|fool@motleyfool.com                       |417       |
|nytdirect@nytimes.com                     |348       |
|ecenter@williams.com                      |340       |
|pmadpr@worldnet.att.net                   |317       |
+------------------------------------------+----------+



### Group B - Question 2: Who are the top 10 recipients by number of emails received (based on the ‘To’ field, filtering out emails not containing this info), along with the number of emails they received

In [0]:
from pyspark.sql.functions import col, explode, split, count

# Filter out rows where 'To' field is null or empty
emails_with_to = df.filter(col("To").isNotNull() & (col("To") != ""))


# Split multiple recipients (assuming they are separated by commas or semicolons)
emails_with_to = emails_with_to.withColumn("Recipient", explode(split(col("To"), "[,;]")))

# Count the number of emails per recipient
recipient_counts = emails_with_to.groupBy("Recipient").agg(count("*").alias("EmailCount"))

# Get the top 10 recipients
top_10_recipients = recipient_counts.orderBy(col("EmailCount").desc()).limit(10)

# Show the result
top_10_recipients.show()

+--------------------+----------+
|           Recipient|EmailCount|
+--------------------+----------+
|                    |     89767|
|pete.davis@enron.com|      9158|
|tana.jones@enron.com|      6244|
|jeff.dasovich@enr...|      5874|
|sara.shackleton@e...|      5867|
|steven.kean@enron...|      5595|
|   vkaminski@aol.com|      4904|
|mark.taylor@enron...|      4622|
|  kay.mann@enron.com|      3819|
|richard.shapiro@e...|      3678|
+--------------------+----------+



### Group B - Question 3: What is the number of emails sent internally within Enron (based on both 'From' and 'To' fields containing @enron.com)

In [0]:
from pyspark.sql.functions import col

# Filter emails where both 'From' and 'To' fields contain '@enron.com'
internal_emails = df.filter(
    (col("From").contains("@enron.com")) & (col("To").contains("@enron.com"))
)

# Count the number of such emails
internal_email_count = internal_emails.count()

print(f"Number of internal emails within Enron: {internal_email_count}")
     

Number of internal emails within Enron: 348376
